# CodeDrift Arena — GRPO Training Notebook

Trains a Qwen2.5-1.5B model to catch stale references in code reviews using GRPO reinforcement learning.

**Runtime:** GPU (T4 or better) — Runtime > Change runtime type > T4 GPU

**Steps:**
1. Clone repo
2. Install dependencies
3. Run the diagnostic (verify pipeline is healthy)
4. Run V2 GRPO training
5. Plot reward curve
6. Test trained model (Battle: Junior vs Senior)


In [ ]:
# Step 1 — Clone the repo
!git clone https://github.com/bansalbhunesh/codedrift-arena.git
%cd codedrift-arena

In [ ]:
# Step 2 — Install training dependencies
!pip install -r requirements-train.txt -q

In [ ]:
# Step 3 — Diagnostic (quick, no model load)
# Verifies: env builds, gold response scores >0.5, GRPO math is healthy
!python scripts/diagnose_v2.py --quick

In [ ]:
# Step 4a — Full diagnostic (loads model, ~2 min)
# Confirms: prompt fits 2048 tokens, model generates valid JSON, reward std > 0.05
!python scripts/diagnose_v2.py --full

In [ ]:
# Step 4b — Run V2 GRPO training
#
# Key flags (all five fixes applied):
#   --max_prompt_length 2048   : prompt is ~1600 tokens; 1024 truncates context
#   --max_completion_length 512: full JSON response needs 200-400 tokens
#   --sft_warmup_steps 50      : teach JSON format before GRPO starts
#   --num_generations 8        : more completions = higher reward variance = real gradient
#   --temperature 1.0          : prevents reward collapse to std=0
#
!python training_v2/train_v2.py 
  --difficulty easy 
  --steps 200 
  --sft_warmup_steps 50 
  --num_generations 8 
  --temperature 1.0 
  --max_prompt_length 2048 
  --max_completion_length 512 
  --backend hf 
  --output_dir outputs/v2_run

In [ ]:
# Step 5 — Plot reward curve from training logs
import matplotlib.pyplot as plt
import json
from pathlib import Path

history_path = Path("outputs/v2_run") / "trainer_state.json"
if history_path.exists():
    state = json.loads(history_path.read_text(encoding="utf-8"))
    log_hist = state.get("log_history", [])
    steps, rewards, stds = [], [], []
    for row in log_hist:
        if "reward" in row and "step" in row:
            steps.append(row["step"])
            rewards.append(row["reward"])
            stds.append(row.get("reward_std", 0))
    if steps:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(steps, rewards, color="#2ecc71")
        ax1.set_title("Reward Mean")
        ax1.set_xlabel("Training step")
        ax1.set_ylabel("Reward")
        ax1.grid(alpha=0.3)
        ax2.plot(steps, stds, color="#e74c3c")
        ax2.set_title("Reward Std (>0.05 = healthy gradient)")
        ax2.set_xlabel("Training step")
        ax2.set_ylabel("Std")
        ax2.grid(alpha=0.3)
        plt.tight_layout()
        Path("outputs").mkdir(exist_ok=True)
        plt.savefig("outputs/reward_curve.png", dpi=150, bbox_inches="tight")
        print("Saved to outputs/reward_curve.png")
        plt.show()
    else:
        print("No reward rows found — check logging_steps setting")
else:
    print(f"trainer_state.json not found at {history_path}")

In [ ]:
# Step 6 — Battle: Junior (untrained) vs Senior (trained)
# Shows the exact reward gap the training produces
import sys
sys.path.insert(0, ".")

from env.codedrift_env import CodeDriftEnv

JUNIOR  = "VERDICT: APPROVE
ROOT_CAUSE: none
ISSUES: none
REASON: Looks fine."
SEED = 42

env_j = CodeDriftEnv(difficulty="easy", seed=SEED)
env_j.reset()
_, rj, _, _ = env_j.step(JUNIOR)

from hf_space.space_app import _trained_response_for
env_s = CodeDriftEnv(difficulty="easy", seed=SEED)
env_s.reset()
senior = _trained_response_for(env_s)
_, rs, _, _ = env_s.step(senior)

print(f"Junior  reward: {rj:+.3f}")
print(f"Senior  reward: {rs:+.3f}")
print(f"Delta:          {rs - rj:+.3f}")